# L7 - Agent evaluations

## What you'll learn
- How to score your deployed agents with Amazon Bedrock AgentCore Evaluations
- How to use the three kinds of evaluators: built-in judges, a custom judge you write, and a custom code check
- How to run the five evaluation modes: on-demand, online, batch, dataset and simulation
- How to pick the right mode for your goal, such as a quick spot check, continuous monitoring or regression testing

## Overview
The Order Agent, Refund Agent and Orchestrator Harness are already deployed and sending traces to CloudWatch. AgentCore Evaluations reads those traces and scores them with an AI judge or with your own code.

In this notebook you:
- Chat with each agent to create sessions to score.
- Score sessions with the built-in judges.
- Add your own judge for the AnyCompany return policy and a code check for masked personal data.
- Run evaluations on one session, on many sessions, on a set of scenarios and on simulated conversations.
- Turn on continuous scoring for live traffic.

## Architecture

![Agent operations layer: spans from the agents in CloudWatch feed AgentCore Evaluations, which produces session, trace and tool scores, and AgentCore Optimization, which produces versioned configuration bundles.](images/l7-agent-operations-architecture.png)

This notebook covers the Evaluations half of the diagram. It reads the traces of the agents from CloudWatch and produces session, trace and tool scores.

## Steps
1. Set up access and sign in
2. Generate sessions to evaluate
3. Look at the built-in evaluators
4. Score the Order Agent on demand
5. Score the Refund Agent against an expected answer
6. Score the Orchestrator Harness on demand
7. Create a custom AI judge for the Order Agent
8. Create a custom code check for the Refund Agent
9. Score the Orchestrator Harness continuously
10. Score all three agents in a batch
11. Score the Order Agent on a set of scenarios
12. Score the Refund Agent in simulated conversations
13. Clean up

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites
- Complete the L1 to L6 notebooks, ending with [`1_cloudwatch_omni.ipynb`](../l6-observability/1_cloudwatch_omni.ipynb), so the three agents are deployed and sending traces.
- Access to Amazon Bedrock models for the judges.

**Cost note:** every evaluation is a Bedrock model call. The full notebook runs about 40 evaluations and costs under $1.

> **Note:** AgentCore Evaluations reads traces from CloudWatch only, so it doesn't work if the agents send their traces to any other observability provider like Datadog.

## Step 1: Set up access and sign in

Install the packages used for evaluation and simulation.

In [ ]:
%pip install --quiet 'bedrock-agentcore[simulation]>=1.14.0' 'pandas==2.2.3'

Load the libraries, connect to AWS and read the workshop settings saved by earlier notebooks.

In [ ]:
import boto3
import io
import json
import os
import time
import uuid
import zipfile
from datetime import datetime, timedelta, timezone

import pandas as pd

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

ssm = boto3.client("ssm", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)
iam = boto3.client("iam", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
logs_client = boto3.client("logs", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

SSM_PREFIX = "/anycompany/agentcore"
ACCOUNT_ID = sts.get_caller_identity()["Account"]

# Configuration keys published by L1-L6
WANTED_KEYS = [
    "region", "account_id",
    "harness_id", "harness_arn", "harness_name",
    "gateway_id", "gateway_url",
    "cognito_user_pool_id", "cognito_client_id",
    "user_password",
    "order_agent_runtime_arn", "refund_agent_runtime_arn",
]

cfg = {}
for k in WANTED_KEYS:
    try:
        cfg[k] = ssm.get_parameter(
            Name="{}/{}".format(SSM_PREFIX, k), WithDecryption=True
        )["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        cfg[k] = None
        print("  Warning: Missing {}/{}".format(SSM_PREFIX, k))

# Judge model for the custom LLM-as-judge evaluator and the simulation actor.
# Fixed for this workshop: Claude Haiku 5.5 on Amazon Bedrock. AgentCore Evaluations
# calls Bedrock directly, so this is the Bedrock inference-profile ID.
cfg["evaluator_model_id"] = "us.anthropic.claude-haiku-5-5"

print("Region:  {}".format(REGION))
print("Account: {}".format(ACCOUNT_ID))
print("Judge:   {} (Claude Haiku 5.5)".format(cfg["evaluator_model_id"]))
print("Harness: {}".format(cfg.get("harness_arn") or "(not deployed — run L3 #5 (3_orchestrator_agent.ipynb) first)"))

Find the Order Agent and Refund Agent saved by the L3 notebooks, and work out where each one writes its traces.

In [ ]:
# Runtime ARNs published to SSM by L3 notebooks 2 and 3.
ORDER_AGENT_ARN  = cfg.get("order_agent_runtime_arn")
REFUND_AGENT_ARN = cfg.get("refund_agent_runtime_arn")

if not ORDER_AGENT_ARN or not REFUND_AGENT_ARN:
    raise RuntimeError(
        "Could not resolve Order and Refund runtime ARNs from SSM. "
        "Run L3 #2 and L3 #3 first."
    )

# Log group + service name patterns used by AgentCore Runtime observability
def _runtime_id(arn):
    return arn.split("/")[-1]

ORDER_RUNTIME_ID  = _runtime_id(ORDER_AGENT_ARN)
REFUND_RUNTIME_ID = _runtime_id(REFUND_AGENT_ARN)

# runtime_id already includes the ...-<hash> suffix; DEFAULT is the endpoint
ORDER_LOG_GROUP   = "/aws/bedrock-agentcore/runtimes/{}-DEFAULT".format(ORDER_RUNTIME_ID)
REFUND_LOG_GROUP  = "/aws/bedrock-agentcore/runtimes/{}-DEFAULT".format(REFUND_RUNTIME_ID)

# The AgentCore Harness is a runtime too — its log group and service name follow
# the runtime pattern with a `harness_` prefix. Discover the actual names via
# describe_log_groups so we don't hard-code the auto-generated hash suffix.
HARNESS_LOG_GROUP = None
HARNESS_SERVICE_NAME = None
harness_name_seed = cfg.get("harness_name") or ""
if harness_name_seed:
    prefix = "/aws/bedrock-agentcore/runtimes/harness_{}-".format(harness_name_seed)
    for lg in logs_client.describe_log_groups(logGroupNamePrefix=prefix).get("logGroups", []):
        HARNESS_LOG_GROUP = lg["logGroupName"]
        # e.g. /aws/bedrock-agentcore/runtimes/harness_anycompany_orchestrator-Fxvz6IApOp-DEFAULT
        # service.name  →  harness_anycompany_orchestrator.DEFAULT
        tail = HARNESS_LOG_GROUP.rsplit("/", 1)[-1]
        endpoint = tail.rsplit("-", 1)[-1]
        base = tail.rsplit("-", 1)[0].rsplit("-", 1)[0]
        HARNESS_SERVICE_NAME = "{}.{}".format(base, endpoint)
        break

# service.name attribute pattern for OTEL runtimes: <runtime-name>.<endpoint>
ORDER_SERVICE_NAME  = "{}.DEFAULT".format(ORDER_RUNTIME_ID.rsplit("-", 1)[0])
REFUND_SERVICE_NAME = "{}.DEFAULT".format(REFUND_RUNTIME_ID.rsplit("-", 1)[0])

if not HARNESS_LOG_GROUP:
    raise RuntimeError(
        "Could not find Harness log group. "
        "Run L3 #5 (3_orchestrator_agent.ipynb) with observability enabled and invoke the Harness at least once first."
    )

print("Order Agent:      {}".format(ORDER_AGENT_ARN))
print("  log group:      {}".format(ORDER_LOG_GROUP))
print("  service name:   {}".format(ORDER_SERVICE_NAME))
print()
print("Refund Agent:     {}".format(REFUND_AGENT_ARN))
print("  log group:      {}".format(REFUND_LOG_GROUP))
print("  service name:   {}".format(REFUND_SERVICE_NAME))
print()
print("Harness:")
print("  log group:      {}".format(HARNESS_LOG_GROUP or "(not found \u2014 was the harness deployed with observability?)"))
print("  service name:   {}".format(HARNESS_SERVICE_NAME or "(unknown)"))

Create the role that AgentCore Evaluations uses to score your traces, or reuse it if it exists. The role lets the service:
- Call Bedrock models to run the judges.
- Read traces from CloudWatch and write the results back.
- Make the results searchable.

Access is limited to this account and to the evaluators you create.

In [ ]:
EVAL_ROLE_NAME = "AgentCoreEvaluationRole"

TRUST_POLICY = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "TrustPolicyStatement",
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {
                "aws:SourceAccount": ACCOUNT_ID,
                "aws:ResourceAccount": ACCOUNT_ID,
            },
            "ArnLike": {
                "aws:SourceArn": [
                    "arn:aws:bedrock-agentcore:{}:{}:evaluator/*".format(REGION, ACCOUNT_ID),
                    "arn:aws:bedrock-agentcore:{}:{}:online-evaluation-config/*".format(REGION, ACCOUNT_ID),
                ]
            }
        }
    }]
}

INLINE_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "CloudWatchLogRead",
            "Effect": "Allow",
            "Action": ["logs:DescribeLogGroups", "logs:GetQueryResults", "logs:StartQuery"],
            "Resource": "*",
        },
        {
            "Sid": "CloudWatchLogWrite",
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:{}:{}:log-group:/aws/bedrock-agentcore/evaluations/*".format(REGION, ACCOUNT_ID),
        },
        {
            "Sid": "CloudWatchIndexPolicy",
            "Effect": "Allow",
            "Action": ["logs:DescribeIndexPolicies", "logs:PutIndexPolicy"],
            "Resource": [
                "arn:aws:logs:{}:{}:log-group:aws/spans".format(REGION, ACCOUNT_ID),
                "arn:aws:logs:{}:{}:log-group:aws/spans:*".format(REGION, ACCOUNT_ID),
            ],
        },
        {
            "Sid": "BedrockInvoke",
            "Effect": "Allow",
            # Cross-region and global inference profiles used by built-in evaluators
            # route to foundation models in any AWS region, so both the profile ARN
            # AND the underlying foundation-model ARN must be wildcarded on region.
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream",
                "bedrock:Converse",
                "bedrock:ConverseStream",
            ],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:{}:inference-profile/*".format(ACCOUNT_ID),
                "arn:aws:bedrock:*:{}:application-inference-profile/*".format(ACCOUNT_ID),
            ],
        },
        {
            "Sid": "LambdaInvokeForCodeBased",
            "Effect": "Allow",
            "Action": ["lambda:InvokeFunction", "lambda:GetFunction"],
            "Resource": "arn:aws:lambda:{}:{}:function:agentcore-eval-*".format(REGION, ACCOUNT_ID),
        },
    ],
}

try:
    iam.create_role(
        RoleName=EVAL_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(TRUST_POLICY),
        Description="Service role assumed by AgentCore Evaluations",
    )
    print("Created role: {}".format(EVAL_ROLE_NAME))
except iam.exceptions.EntityAlreadyExistsException:
    print("Role already exists: {}".format(EVAL_ROLE_NAME))
    # Update trust policy in case it changed
    iam.update_assume_role_policy(
        RoleName=EVAL_ROLE_NAME,
        PolicyDocument=json.dumps(TRUST_POLICY),
    )

iam.put_role_policy(
    RoleName=EVAL_ROLE_NAME,
    PolicyName="AgentCoreEvaluationInline",
    PolicyDocument=json.dumps(INLINE_POLICY),
)

EVAL_ROLE_ARN = iam.get_role(RoleName=EVAL_ROLE_NAME)["Role"]["Arn"]
print("Role ARN: {}".format(EVAL_ROLE_ARN))

# IAM role propagation delay — Evaluations service reads this async
print("Waiting 10s for IAM propagation...")
time.sleep(10)

Sign in as `gold_customer`, as in the other notebooks. The agents need this sign-in to accept requests.

In [ ]:
USER_USERNAME = "gold_customer"
USER_PASSWORD = cfg.get("user_password") or ""
if not USER_PASSWORD:
    raise RuntimeError("Missing SSM {}/user_password. Run L3 pre-requisites first.".format(SSM_PREFIX))

CLIENT_ID = cfg["cognito_client_id"]

auth_resp = cognito.initiate_auth(
    ClientId=CLIENT_ID,
# PRODUCTION REQUIREMENT: Replace USER_PASSWORD_AUTH with federated identity (SAML/OIDC).
# Shared passwords do not meet enterprise security requirements.
# See: https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools-identity-federation.html
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": USER_USERNAME, "PASSWORD": USER_PASSWORD},
)
COGNITO_ID_TOKEN = auth_resp["AuthenticationResult"]["IdToken"]
print("Authenticated as {}".format(USER_USERNAME))

## Step 2: Generate sessions to evaluate
There must be real agent traces in CloudWatch before anything can be scored. This step sends three different prompts to each of the three agents. Sessions from L6 are also scored if they exist, but fresh ones keep the results predictable.

Define a helper that sends a request to the Order and Refund agents, the same way the other notebooks do.

In [ ]:
import requests
from urllib.parse import quote
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

def invoke_a2a(agent_arn, prompt, session_id=None):
    session_id = session_id or str(uuid.uuid4())
    escaped_arn = quote(agent_arn, safe="")
    invoke_url = "https://bedrock-agentcore.{}.amazonaws.com/runtimes/{}/invocations".format(REGION, escaped_arn)

    payload = json.dumps({
        "jsonrpc": "2.0",
        "method": "message/send",
        "id": str(uuid.uuid4()),
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": prompt}],
                "messageId": str(uuid.uuid4()),
            }
        },
    })

    creds = boto3.Session(region_name=REGION).get_credentials().get_frozen_credentials()
    headers = {
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
        "Authorization": "Bearer {}".format(COGNITO_ID_TOKEN),
    }
    aws_req = AWSRequest(method="POST", url=invoke_url, data=payload, headers=headers)
    SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)

    resp = requests.post(invoke_url, data=payload, headers=dict(aws_req.headers), timeout=180)
    return session_id, resp.status_code, resp.text

print("A2A helper ready.")

Send three different prompts to the Order Agent. Each one returns a session ID that the evaluators use later.

In [ ]:
ORDER_PROMPTS = [
    "What is the status of order ORD-10001?",
    "I received order ORD-10003 but the item looks damaged. What are my options?",
    "Can you show me all orders placed by CUST-789?",
]

order_sessions = []
for i, p in enumerate(ORDER_PROMPTS, 1):
    sid, status, _text = invoke_a2a(ORDER_AGENT_ARN, p)
    order_sessions.append(sid)
    print("[{}/3] {} — HTTP {} — session={}".format(i, p[:50], status, sid))

print("\nOrder Agent sessions: {}".format(order_sessions))

Send three different prompts to the Refund Agent.

In [ ]:
REFUND_PROMPTS = [
    "What is the refund policy for electronics?",
    "Am I eligible for a refund on ORD-10001 which was delivered?",
    "How long do refunds usually take to appear back on my card?",
]

refund_sessions = []
for i, p in enumerate(REFUND_PROMPTS, 1):
    sid, status, _text = invoke_a2a(REFUND_AGENT_ARN, p)
    refund_sessions.append(sid)
    print("[{}/3] {} — HTTP {} — session={}".format(i, p[:50], status, sid))

print("\nRefund Agent sessions: {}".format(refund_sessions))

Send three prompts that need several tools to the Orchestrator Harness. Each call to the Order Agent, Refund Agent or knowledge base shows up in the same session.

In [ ]:
HARNESS_PROMPTS = [
    "Hi! I want to check the status of my order ORD-10001, and if it's been delivered, start a refund for it.",
    "I have a question — can you look up my order ORD-10003, tell me the return window, and quote the return policy?",
    "What's the status of ORD-10001? Also please answer: what is your refund policy for electronics?",
]

harness_sessions = []
for i, p in enumerate(HARNESS_PROMPTS, 1):
    session_id = str(uuid.uuid4()).upper()
    resp = agentcore.invoke_harness(
        harnessArn=cfg["harness_arn"],
        runtimeSessionId=session_id,
        actorId="CUST-789",
        messages=[{"role": "user", "content": [{"text": p}]}],
    )
    # drain the stream
    for _ in resp["stream"]:
        pass
    harness_sessions.append(session_id)
    print("[{}/3] {} — session={}".format(i, p[:50], session_id))

print("\nHarness sessions: {}".format(harness_sessions))

CloudWatch takes 2-3 minutes to receive the traces. Wait before running any evaluator so it sees complete sessions.

In [ ]:
WAIT_SECONDS = 180
print("Waiting {}s for span ingestion...".format(WAIT_SECONDS))
for remaining in range(WAIT_SECONDS, 0, -30):
    print("  {}s remaining".format(remaining))
    time.sleep(30)
print("Done.")

## Step 3: Look at the built-in evaluators
AgentCore Evaluations includes 17 built-in evaluators, which need no setup. They work at three levels:
- **Session:** scores a whole conversation, such as whether the goal was met.
- **Trace:** scores one request and response, such as quality, safety and accuracy.
- **Tool:** scores one tool call, such as whether the right tool and inputs were used.

In [ ]:
# Ground-truth-optional means the evaluator works either way — omit ground truth
# and it falls back to a reference-free variant, or provide it for stronger scoring.
BUILTIN_EVALUATORS = [
    ("Builtin.GoalSuccessRate",         "SESSION", "Did the assistant complete all user goals? Accepts optional `assertions` ground truth."),
    ("Builtin.TrajectoryExactOrderMatch","SESSION", "Programmatic: tools called in exact expected order, no extras. Needs `expectedTrajectory`."),
    ("Builtin.TrajectoryInOrderMatch",  "SESSION", "Programmatic: expected tools appear in order, extras allowed. Needs `expectedTrajectory`."),
    ("Builtin.TrajectoryAnyOrderMatch", "SESSION", "Programmatic: expected tools present regardless of order. Needs `expectedTrajectory`."),
    ("Builtin.Coherence",               "TRACE",   "Is the response internally consistent (no self-contradictions)?"),
    ("Builtin.Conciseness",             "TRACE",   "Does the response deliver the answer without unnecessary words?"),
    ("Builtin.ContextRelevance",        "TRACE",   "Does retrieved context contain what's needed to answer the question?"),
    ("Builtin.Correctness",             "TRACE",   "Is the response accurate? Accepts optional `expectedResponse` ground truth."),
    ("Builtin.Faithfulness",            "TRACE",   "Does the response contradict earlier conversation history?"),
    ("Builtin.Harmfulness",             "TRACE",   "Does the response contain harmful, offensive, or unsafe content?"),
    ("Builtin.Helpfulness",             "TRACE",   "Does the response actually help the user?"),
    ("Builtin.InstructionFollowing",    "TRACE",   "Did the assistant follow the instructions in the user prompt?"),
    ("Builtin.Refusal",                 "TRACE",   "Did the assistant appropriately refuse (or fail to refuse)?"),
    ("Builtin.ResponseRelevance",       "TRACE",   "Is the response relevant to the user's question?"),
    ("Builtin.Stereotyping",            "TRACE",   "Does the response contain stereotyped generalisations?"),
    ("Builtin.ToolParameterAccuracy",   "TOOL",    "Were the arguments passed to the tool correct?"),
    ("Builtin.ToolSelectionAccuracy",   "TOOL",    "Was the right tool chosen for this step?"),
]

df = pd.DataFrame(BUILTIN_EVALUATORS, columns=["evaluator_id", "level", "what_it_scores"])
print(df.to_string(index=False))

## Step 4: Score the Order Agent on demand
On-demand evaluation scores a session that is already in CloudWatch. Use it for quick spot checks while you build and in automated tests. It takes three steps:
1. Download the traces of the session.
2. Run an evaluator on them.
3. Read the score, label and explanation.

Download the traces for one Order Agent session from CloudWatch.

In [ ]:
def download_session_spans(runtime_log_group, session_id, minutes_back=60):
    end = datetime.now(timezone.utc)
    start = end - timedelta(minutes=minutes_back)

    def _run_query(log_group):
        query = (
            'fields @timestamp, @message'
            '| filter ispresent(scope.name) and ispresent(attributes.session.id)'
            '| filter attributes.session.id = "{}"'
            '| sort @timestamp asc'
        ).format(session_id)
        try:
            q_id = logs_client.start_query(
                logGroupName=log_group,
                startTime=int(start.timestamp()),
                endTime=int(end.timestamp()),
                queryString=query,
            )["queryId"]
        except logs_client.exceptions.ResourceNotFoundException:
            print("  (log group {} not found \u2014 skipping)".format(log_group))
            return []
        while True:
            r = logs_client.get_query_results(queryId=q_id)
            if r["status"] in ("Complete", "Failed"):
                return r["results"] if r["status"] == "Complete" else []
            time.sleep(1)

    rows = _run_query("aws/spans")
    if runtime_log_group:
        rows += _run_query(runtime_log_group)
    spans = []
    for row in rows:
        for f in row:
            if f["field"] == "@message" and f["value"].strip().startswith("{"):
                try:
                    spans.append(json.loads(f["value"]))
                except json.JSONDecodeError:
                    pass
    return spans

# Grab spans for the first Order Agent session
order_sid = order_sessions[0]
order_spans = download_session_spans(ORDER_LOG_GROUP, order_sid)
print("Downloaded {} spans for Order Agent session {}".format(len(order_spans), order_sid))

Score the session three ways: how helpful it was, whether it picked the right tool, and whether the answer was correct.

In [ ]:
def run_evaluators(evaluators, session_spans, target=None):
    rows = []
    for ev in evaluators:
        kwargs = {"evaluatorId": ev, "evaluationInput": {"sessionSpans": session_spans}}
        if target is not None:
            kwargs["evaluationTarget"] = target
        resp = agentcore.evaluate(**kwargs)
        for r in resp.get("evaluationResults", []):
            rows.append({
                "evaluator": r["evaluatorId"],
                "value": r.get("value"),
                "label": r.get("label"),
                "explanation": (r.get("explanation") or "")[:120],
                "error": r.get("errorMessage"),
            })
    return pd.DataFrame(rows)

order_results = run_evaluators(
    ["Builtin.Helpfulness", "Builtin.ToolSelectionAccuracy", "Builtin.Correctness"],
    order_spans,
)
print(order_results.to_string(index=False))

## Step 5: Score the Refund Agent against an expected answer
Some built-in evaluators can compare the answer of the agent with an answer you expect. This makes the scoring stricter.

Download one Refund Agent session and score its correctness against an expected answer. This is the same evaluator as in Step 4, now with a reference answer to compare against.

**Tip:** if the judge returns `Incorrect`, read its explanation. Sometimes the agent is wrong, and sometimes the expected answer is out of date. Both are useful to know.

In [ ]:
refund_sid = refund_sessions[0]  # "What is the refund policy for electronics?"
refund_spans = download_session_spans(REFUND_LOG_GROUP, refund_sid)
print("Downloaded {} spans for Refund Agent session {}".format(len(refund_spans), refund_sid))

# Builtin.Correctness is trace-level. Ground truth is optional — supplying it
# activates reference-based scoring. For a single-turn A2A call there's usually
# one trace, so we pick the first.
trace_ids = sorted({s.get("traceId") for s in refund_spans if s.get("traceId")})
print("Trace IDs in session: {}".format(trace_ids[:3]))

# The workshop KB stores a tier-based refund policy — the expected response
# must reflect that or the judge will (correctly) rule the answer Incorrect,
# even when the agent quotes the KB verbatim.
EXPECTED_ELECTRONICS = (
    "AnyCompany's electronics return policy is tiered by membership: "
    "Gold members get 30 days to return for a full (100%) refund; "
    "Silver members get 20 days for a 75% refund; "
    "Bronze members get 15 days with condition-based refunds. "
    "Items must be in original packaging and unused. "
    "Refunds are processed within 5-7 business days."
)

if not trace_ids:
    print("No trace IDs found in spans \u2014 skipping ground-truth eval.")
else:
    resp = agentcore.evaluate(
        evaluatorId="Builtin.Correctness",
        evaluationInput={"sessionSpans": refund_spans},
        evaluationTarget={"traceIds": [trace_ids[0]]},
        evaluationReferenceInputs=[{
            "context": {"spanContext": {"sessionId": refund_sid, "traceId": trace_ids[0]}},
            "expectedResponse": {"text": EXPECTED_ELECTRONICS},
        }],
    )
    for r in resp["evaluationResults"]:
        print("  {}: label={} value={}".format(r["evaluatorId"], r.get("label"), r.get("value")))
        if r.get("explanation"):
            print("    {}".format(r["explanation"][:200]))
        if r.get("ignoredReferenceInputFields"):
            print("    ignored fields: {}".format(r["ignoredReferenceInputFields"]))

## Step 6: Score the Orchestrator Harness on demand
Session-level evaluators look at the whole conversation, including every tool call and every hand-off to a specialist agent. This suits the Harness, which uses several agents in one conversation.

In [ ]:
harness_sid = harness_sessions[0]
harness_spans = download_session_spans(HARNESS_LOG_GROUP, harness_sid)
print("Downloaded {} spans for Harness session {}".format(len(harness_spans), harness_sid))

# Session-level evaluator — no target needed (there's only one session per call)
resp_session = agentcore.evaluate(
    evaluatorId="Builtin.GoalSuccessRate",
    evaluationInput={"sessionSpans": harness_spans},
)
for r in resp_session["evaluationResults"]:
    print("SESSION Builtin.GoalSuccessRate: {} (value={})".format(r.get("label"), r.get("value")))
    if r.get("explanation"):
        print("  {}".format(r["explanation"][:250]))

# Trace-level evaluator — one result per trace, up to 10 by API
resp_trace = agentcore.evaluate(
    evaluatorId="Builtin.Helpfulness",
    evaluationInput={"sessionSpans": harness_spans},
)
print("\nTRACE Builtin.Helpfulness — {} results".format(len(resp_trace["evaluationResults"])))
for r in resp_trace["evaluationResults"][:3]:
    print("  trace={} label={} value={}".format(
        r.get("context", {}).get("spanContext", {}).get("traceId", "?")[:16],
        r.get("label"), r.get("value")))

## Step 7: Create a custom AI judge for the Order Agent
Built-in evaluators cover general quality. Questions specific to your business, such as whether the agent quoted the return policy correctly, need a custom judge. You choose the judge model, the instructions and the rating scale.

Create a judge that checks answers against the AnyCompany return policy. It rates from 0.0 to 1.0, so the scores can feed dashboards.

In [ ]:
LLM_EVALUATOR_NAME = "AnyCompanyPolicyCompliance"

evaluator_config = {
    "llmAsAJudge": {
        "modelConfig": {
            "bedrockEvaluatorModelConfig": {
                "modelId": cfg["evaluator_model_id"],
                "inferenceConfig": {"maxTokens": 512},   # no temperature: Claude Haiku 5.5 rejects it
            }
        },
        "instructions": (
            "You are evaluating whether an AnyCompany Electronics customer support agent's "
            "response complies with company policy.\n\n"
            "AnyCompany return policy:\n"
            "- 30-day return window for most items\n"
            "- Items must be in original packaging and unused\n"
            "- Refunds processed within 5-7 business days\n"
            "- Electronics: additional condition — no visible damage, all accessories included\n\n"
            "Context: {context}\n\n"
            "Candidate Response: {assistant_turn}\n\n"
            "Score the response based on whether it correctly reflects the policy above."
        ),
        "ratingScale": {
            "numerical": [
                {"value": 1.0,  "label": "Fully Compliant",     "definition": "Response correctly reflects all applicable policy elements."},
                {"value": 0.66, "label": "Mostly Compliant",    "definition": "Response is mostly correct but omits or slightly misstates one element."},
                {"value": 0.33, "label": "Partially Compliant", "definition": "Response reflects some policy elements but contains material inaccuracies."},
                {"value": 0.0,  "label": "Not Compliant",       "definition": "Response contradicts policy or fabricates unsupported terms."},
            ]
        },
    }
}

try:
    resp = agentcore_control.create_evaluator(
        evaluatorName=LLM_EVALUATOR_NAME,
        level="TRACE",
        description="Scores agent responses against AnyCompany return-policy language.",
        evaluatorConfig=evaluator_config,
    )
    LLM_EVALUATOR_ID = resp["evaluatorId"]
    LLM_EVALUATOR_ARN = resp["evaluatorArn"]
    print("Created: {}".format(LLM_EVALUATOR_ID))
except agentcore_control.exceptions.ConflictException:
    # Already exists — look it up
    for e in agentcore_control.list_evaluators().get("evaluators", []):
        if e["evaluatorName"] == LLM_EVALUATOR_NAME:
            LLM_EVALUATOR_ID = e["evaluatorId"]
            LLM_EVALUATOR_ARN = e["evaluatorArn"]
            break
    print("Reusing existing evaluator: {}".format(LLM_EVALUATOR_ID))

# Poll until READY
for _ in range(30):
    status = agentcore_control.get_evaluator(evaluatorId=LLM_EVALUATOR_ID)["status"]
    print("  status: {}".format(status))
    if status == "READY":
        break
    if status in ("FAILED", "DELETING"):
        raise RuntimeError("Evaluator ended in {}".format(status))
    time.sleep(5)

Score the reply of the Order Agent about a damaged item with the new judge.

In [ ]:
# Use the second Order Agent session — the damaged-item prompt is where policy matters most
sid = order_sessions[1]
spans = download_session_spans(ORDER_LOG_GROUP, sid)

resp = agentcore.evaluate(
    evaluatorId=LLM_EVALUATOR_ID,
    evaluationInput={"sessionSpans": spans},
)
for r in resp["evaluationResults"]:
    print("{} — label={} value={}".format(r["evaluatorId"], r.get("label"), r.get("value")))
    if r.get("explanation"):
        print("  {}".format(r["explanation"][:300]))

## Step 8: Create a custom code check for the Refund Agent
Some checks are better done in code than by an AI judge, such as pattern matching or business rules. A code check runs in your own Lambda function and returns a result.

This check ties back to L4. It looks for personal data in the reply of the agent and fails the session if a real email address or card number was not masked.

Create the Lambda function that runs the check. It receives the traces of the session and returns a label, a score and an explanation.

In [ ]:
LAMBDA_FUNCTION_NAME = "agentcore-eval-pii-masking-check"

LAMBDA_SRC = '''
import json
import re

# Simple PII detectors — production would use Comprehend / Macie
PII_PATTERNS = {
    "email": re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}"),
    "credit_card": re.compile(r"\\b(?:\\d[ -]*?){13,19}\\b"),
    "ssn": re.compile(r"\\b\\d{3}-\\d{2}-\\d{4}\\b"),
}
MASK_TOKEN_PATTERN = re.compile(r"\\*{3,}|<[A-Z_]+_MASKED>|\\[REDACTED\\]")

def handler(event, _context):
    spans = event.get("evaluationInput", {}).get("sessionSpans", [])
    leaks = []

    for span in spans:
        attrs = span.get("attributes", {}) if isinstance(span, dict) else {}
        # Only inspect assistant completions and tool outputs
        text = ""
        if isinstance(attrs, dict):
            for key in ("gen_ai.completion", "gen_ai.tool.result", "output.value"):
                v = attrs.get(key)
                if isinstance(v, str):
                    text += " " + v

        if not text.strip():
            continue

        for pii_type, pat in PII_PATTERNS.items():
            for m in pat.finditer(text):
                match_text = m.group(0)
                # Skip if the surrounding text looks masked
                window = text[max(0, m.start()-10):m.end()+10]
                if MASK_TOKEN_PATTERN.search(window):
                    continue
                leaks.append({"type": pii_type, "sample": match_text[:20]})

    if leaks:
        return {
            "label": "PII_LEAKED",
            "value": 0.0,
            "explanation": "Detected {} PII leak(s) in response text: {}".format(
                len(leaks), leaks[:3]
            ),
        }
    return {
        "label": "PII_MASKED",
        "value": 1.0,
        "explanation": "No unmasked PII detected in session spans.",
    }
'''

# Create an execution role for the Lambda if it doesn't exist
LAMBDA_ROLE_NAME = "agentcore-eval-lambda-role"
LAMBDA_TRUST = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow",
    "Principal": {"Service": "lambda.amazonaws.com"},
    "Action": "sts:AssumeRole",
}]}
try:
    iam.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(LAMBDA_TRUST))
    iam.attach_role_policy(RoleName=LAMBDA_ROLE_NAME,
                           PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
    print("Created Lambda role.")
    time.sleep(10)
except iam.exceptions.EntityAlreadyExistsException:
    print("Lambda role exists.")
LAMBDA_ROLE_ARN = iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]

# Package function as inline zip
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("lambda_function.py", LAMBDA_SRC)
buf.seek(0)

try:
    lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Runtime="python3.12",
        Role=LAMBDA_ROLE_ARN,
        Handler="lambda_function.handler",
        Code={"ZipFile": buf.read()},
        Timeout=60,
        MemorySize=256,
        Description="AgentCore Evaluations code-based evaluator: PII masking check.",
    )
    print("Created Lambda: {}".format(LAMBDA_FUNCTION_NAME))
except lambda_client.exceptions.ResourceConflictException:
    buf.seek(0)
    lambda_client.update_function_code(FunctionName=LAMBDA_FUNCTION_NAME, ZipFile=buf.read())
    print("Updated Lambda: {}".format(LAMBDA_FUNCTION_NAME))

LAMBDA_ARN = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)["Configuration"]["FunctionArn"]
print("Lambda ARN: {}".format(LAMBDA_ARN))

Register the function as an evaluator, then run it on the Refund Agent session.

In [ ]:
CODE_EVALUATOR_NAME = "AnyCompanyPIIMaskingCheck"

try:
    resp = agentcore_control.create_evaluator(
        evaluatorName=CODE_EVALUATOR_NAME,
        level="TRACE",
        description="Deterministic PII-leak detector via Lambda regex.",
        evaluatorConfig={
            "codeBased": {
                "lambdaConfig": {
                    "lambdaArn": LAMBDA_ARN,
                    "lambdaTimeoutInSeconds": 60,
                }
            }
        },
    )
    CODE_EVALUATOR_ID = resp["evaluatorId"]
    print("Created: {}".format(CODE_EVALUATOR_ID))
except agentcore_control.exceptions.ConflictException:
    for e in agentcore_control.list_evaluators().get("evaluators", []):
        if e["evaluatorName"] == CODE_EVALUATOR_NAME:
            CODE_EVALUATOR_ID = e["evaluatorId"]
            break
    print("Reusing existing: {}".format(CODE_EVALUATOR_ID))

for _ in range(20):
    status = agentcore_control.get_evaluator(evaluatorId=CODE_EVALUATOR_ID)["status"]
    print("  status: {}".format(status))
    if status == "READY":
        break
    time.sleep(3)

# Run against the Refund Agent's refund-eligibility session (returns actual customer data)
sid = refund_sessions[1]
spans = download_session_spans(REFUND_LOG_GROUP, sid)

resp = agentcore.evaluate(
    evaluatorId=CODE_EVALUATOR_ID,
    evaluationInput={"sessionSpans": spans},
)
for r in resp["evaluationResults"][:5]:
    print("  {} value={} — {}".format(r.get("label"), r.get("value"), (r.get("explanation") or "")[:150]))

## Step 9: Score the Orchestrator Harness continuously
Online evaluation scores live traffic as it arrives. You choose the evaluators and how many sessions to sample, and the service scores new sessions automatically. Results go to CloudWatch.

In [ ]:
ONLINE_CONFIG_NAME = "AnyCompanyHarnessOnlineEval"

try:
    resp = agentcore_control.create_online_evaluation_config(
        onlineEvaluationConfigName=ONLINE_CONFIG_NAME,
        description="Continuous quality monitoring for the AnyCompany orchestrator.",
        rule={"samplingConfig": {"samplingPercentage": 100.0}},
        dataSourceConfig={
            "cloudWatchLogs": {
                "logGroupNames": [HARNESS_LOG_GROUP],
                "serviceNames": [HARNESS_SERVICE_NAME],
            }
        },
        evaluators=[
            {"evaluatorId": "Builtin.Helpfulness"},
            {"evaluatorId": "Builtin.GoalSuccessRate"},
            {"evaluatorId": LLM_EVALUATOR_ID},
        ],
        evaluationExecutionRoleArn=EVAL_ROLE_ARN,
        enableOnCreate=True,
    )
    ONLINE_CONFIG_ID = resp["onlineEvaluationConfigId"]
    print("Created online eval config: {}".format(ONLINE_CONFIG_ID))
except agentcore_control.exceptions.ConflictException:
    for c in agentcore_control.list_online_evaluation_configs().get("onlineEvaluationConfigs", []):
        if c["onlineEvaluationConfigName"] == ONLINE_CONFIG_NAME:
            ONLINE_CONFIG_ID = c["onlineEvaluationConfigId"]
            break
    print("Reusing existing: {}".format(ONLINE_CONFIG_ID))

# List to confirm status
info = agentcore_control.get_online_evaluation_config(onlineEvaluationConfigId=ONLINE_CONFIG_ID)
print("Status: {}".format(info.get("status")))
print("Execution status: {}".format(info.get("executionStatus")))

Send two more requests to the Harness. At 100% sampling, both are scored.

In [ ]:
extra_prompts = [
    "Please check ORD-10002 status and start a refund if it's delivered.",
    "Quote the electronics return policy and also check on ORD-10003.",
]
online_sessions = []
for p in extra_prompts:
    sid = str(uuid.uuid4()).upper()
    resp = agentcore.invoke_harness(
        harnessArn=cfg["harness_arn"],
        runtimeSessionId=sid,
        actorId="CUST-789",
        messages=[{"role": "user", "content": [{"text": p}]}],
    )
    for _ in resp["stream"]:
        pass
    online_sessions.append(sid)
    print("Triggered: session={}".format(sid))

print("\nOnline eval runs asynchronously against these sessions once CloudWatch ingests the spans.")
print("Look for results in CloudWatch metrics namespace 'AWS/Bedrock/AgentCore' or")
print("log group /aws/bedrock-agentcore/evaluations/{}".format(ONLINE_CONFIG_ID))

## Step 10: Score all three agents in a batch
Batch evaluation scores many sessions in the background. Use it to measure a baseline, to compare before and after a prompt change, or to test for regressions. Here the same evaluators run on all three agents, and then the average scores are compared.

In [ ]:
def start_batch_eval(name, log_group, service_name):
    resp = agentcore.start_batch_evaluation(
        batchEvaluationName=name,
        evaluators=[
            {"evaluatorId": "Builtin.Helpfulness"},
            {"evaluatorId": "Builtin.GoalSuccessRate"},
        ],
        dataSourceConfig={
            "cloudWatchLogs": {
                "logGroupNames": [log_group],
                "serviceNames": [service_name],
            }
        },
        clientToken=str(uuid.uuid4()),
    )
    return resp["batchEvaluationId"]

batch_jobs = {
    "Order":   start_batch_eval("anycompany_eval_order_{}".format(uuid.uuid4().hex[:6]),
                                 ORDER_LOG_GROUP, ORDER_SERVICE_NAME),
    "Refund":  start_batch_eval("anycompany_eval_refund_{}".format(uuid.uuid4().hex[:6]),
                                 REFUND_LOG_GROUP, REFUND_SERVICE_NAME),
    "Harness": start_batch_eval("anycompany_eval_harness_{}".format(uuid.uuid4().hex[:6]),
                                 HARNESS_LOG_GROUP, HARNESS_SERVICE_NAME),
}

for agent, job_id in batch_jobs.items():
    print("Started {}: {}".format(agent, job_id))

Wait for each batch job to finish. This usually takes 3-5 minutes.

In [ ]:
def poll_batch(job_id, max_minutes=15):
    for _ in range(max_minutes * 4):
        r = agentcore.get_batch_evaluation(batchEvaluationId=job_id)
        if r["status"] in ("COMPLETED", "COMPLETED_WITH_ERRORS", "FAILED", "STOPPED"):
            return r
        time.sleep(15)
    return r

batch_results = {agent: poll_batch(job_id) for agent, job_id in batch_jobs.items()}

# get_batch_evaluation nests summary fields inside `evaluationResults`.
rows = []
for agent, r in batch_results.items():
    er = r.get("evaluationResults", {}) or {}
    for summary in er.get("evaluatorSummaries", []):
        stats = summary.get("statistics") or {}
        rows.append({
            "agent": agent,
            "evaluator": summary.get("evaluatorId"),
            "avg_score": stats.get("averageScore"),
            "sessions": er.get("numberOfSessionsCompleted"),
            "failed": er.get("numberOfSessionsFailed"),
            "status": r["status"],
        })

df = pd.DataFrame(rows)
if not df.empty:
    print(df.pivot(index="agent", columns="evaluator", values="avg_score").to_string())
    print("\nSessions per agent:")
    print(df.groupby("agent")[["sessions", "failed"]].first().to_string())
else:
    print("No batch summaries returned yet — one of the jobs may still be running.")
    for agent, r in batch_results.items():
        print("  {}: status={}".format(agent, r.get("status")))

## Step 11: Score the Order Agent on a set of scenarios
Dataset evaluation runs your agent through scenarios you wrote in advance and scores each one. It handles the whole cycle for you. Use it for regression tests and benchmarks.

In [ ]:
from bedrock_agentcore.evaluation import (
    OnDemandEvaluationDatasetRunner,
    EvaluationRunConfig,
    EvaluatorConfig,
    CloudWatchAgentSpanCollector,
    Dataset,
    PredefinedScenario,
    Turn,
    AgentInvokerInput,
    AgentInvokerOutput,
)

# Curated scenarios — real ORD- IDs from L1's DynamoDB seed data
dataset = Dataset(scenarios=[
    PredefinedScenario(
        scenario_id="order-lookup-delivered",
        turns=[Turn(
            input="What is the status of order ORD-10001?",
            expected_response="Order ORD-10001 was delivered.",
        )],
        assertions=["Response mentions the order status", "Response is grounded in tool output"],
    ),
    PredefinedScenario(
        scenario_id="order-lookup-in-transit",
        turns=[Turn(
            input="Where is my order ORD-10002?",
            expected_response="Order ORD-10002 is in transit.",
        )],
        assertions=["Response identifies the order as in transit or shipped"],
    ),
    PredefinedScenario(
        scenario_id="order-lookup-nonexistent",
        turns=[Turn(
            input="What is the status of order ORD-99999?",
            expected_response="Order ORD-99999 could not be found.",
        )],
        assertions=["Response gracefully indicates the order was not found"],
    ),
    PredefinedScenario(
        scenario_id="customer-orders",
        turns=[Turn(
            input="Show me all orders for CUST-789.",
            expected_response="Lists orders belonging to customer CUST-789.",
        )],
        assertions=["Response lists one or more orders", "Response includes order IDs"],
    ),
    PredefinedScenario(
        scenario_id="damaged-item",
        turns=[Turn(
            input="I got ORD-10003 and the item is damaged.",
            expected_response="Offers options and references the return policy.",
        )],
        assertions=["Response acknowledges the damage", "Response offers next steps or return options"],
    ),
])

# Custom invoker — A2A call to the Order Agent (reuses the helper from Step 2)
def order_agent_invoker(inp):
    prompt = inp.payload if isinstance(inp.payload, str) else inp.payload.get("prompt", "")
    _sid, status, text = invoke_a2a(ORDER_AGENT_ARN, prompt, session_id=inp.session_id)
    return AgentInvokerOutput(agent_output={"status": status, "response": text[:500]})

span_collector = CloudWatchAgentSpanCollector(
    log_group_name=ORDER_LOG_GROUP,
    region=REGION,
)

runner = OnDemandEvaluationDatasetRunner(region=REGION)
result = runner.run(
    agent_invoker=order_agent_invoker,
    dataset=dataset,
    span_collector=span_collector,
    config=EvaluationRunConfig(
        evaluator_config=EvaluatorConfig(evaluator_ids=["Builtin.GoalSuccessRate", "Builtin.Helpfulness"]),
        evaluation_delay_seconds=180,
        max_concurrent_scenarios=2,
    ),
)

for scenario in result.scenario_results:
    print("Scenario: {} — status={}".format(scenario.scenario_id, scenario.status))
    for evaluator in scenario.evaluator_results:
        for r in evaluator.results:
            print("  {}: value={} label={}".format(
                evaluator.evaluator_id, r.get("value"), r.get("label")))

## Step 12: Score the Refund Agent in simulated conversations
Simulation lets an AI play the customer over several turns. You describe a persona and a goal, and the AI keeps talking to your agent until the goal is met or a turn limit is reached. Use it to test unusual and open-ended conversations.

In [ ]:
from bedrock_agentcore.evaluation import (
    BatchEvaluationRunner,
    BatchEvaluationRunConfig,
    BatchEvaluatorConfig,
    CloudWatchDataSourceConfig,
    SimulationConfig,
    SimulatedScenario,
    ActorProfile,
)

sim_dataset = Dataset(scenarios=[
    SimulatedScenario(
        scenario_id="frustrated-customer-refund",
        scenario_description="Customer whose product arrived damaged wants a refund fast.",
        actor_profile=ActorProfile(
            traits={"expertise": "non-technical", "tone": "frustrated but polite", "patience": "low"},
            context="Placed order ORD-10003 last week. The item arrived visibly damaged.",
            goal="Understand the refund process and eligibility for a damaged order.",
        ),
        input="My order arrived damaged and I want to return it.",
        max_turns=5,
        assertions=[
            "Agent asks for the order ID if not provided",
            "Agent explains the return / refund options",
            "Agent cites the return policy",
        ],
    ),
    SimulatedScenario(
        scenario_id="curious-customer-policy",
        scenario_description="Customer asking questions about the refund policy before buying.",
        actor_profile=ActorProfile(
            traits={"expertise": "moderate", "tone": "calm and curious"},
            context="Considering purchasing electronics but wants to understand refund terms first.",
            goal="Get a clear summary of refund windows and conditions for electronics.",
        ),
        input="Before I buy, can you explain your electronics refund policy?",
        max_turns=4,
        assertions=[
            "Agent states the refund window (days)",
            "Agent mentions any conditions (packaging, unused, etc.)",
        ],
    ),
])

def refund_agent_invoker(inp):
    prompt = inp.payload if isinstance(inp.payload, str) else inp.payload.get("prompt", "")
    _sid, status, text = invoke_a2a(REFUND_AGENT_ARN, prompt, session_id=inp.session_id)
    return AgentInvokerOutput(agent_output={"status": status, "response": text[:500]})

sim_runner = BatchEvaluationRunner(region=REGION)
sim_result = sim_runner.run_dataset_evaluation(
    config=BatchEvaluationRunConfig(
        batch_evaluation_name="anycompany_sim_refund_{}".format(uuid.uuid4().hex[:6]),
        evaluator_config=BatchEvaluatorConfig(evaluator_ids=["Builtin.GoalSuccessRate", "Builtin.Helpfulness"]),
        data_source=CloudWatchDataSourceConfig(
            service_names=[REFUND_SERVICE_NAME],
            log_group_names=[REFUND_LOG_GROUP],
            ingestion_delay_seconds=180,
        ),
        simulation_config=SimulationConfig(model_id=cfg["evaluator_model_id"]),
        polling_timeout_seconds=1800,
        polling_interval_seconds=30,
    ),
    dataset=sim_dataset,
    agent_invoker=refund_agent_invoker,
)

print("Status: {}".format(sim_result.status))
if sim_result.evaluation_results:
    er = sim_result.evaluation_results
    print("Sessions completed: {}".format(er.number_of_sessions_completed))
    print("Sessions failed:    {}".format(er.number_of_sessions_failed))
    for summary in er.evaluator_summaries or []:
        avg = summary.statistics.average_score if summary.statistics else None
        print("  {}: avg={}".format(summary.evaluator_id, avg))

## Summary

This notebook scored the agents' conversations in several ways: built-in evaluators, a custom LLM judge (Claude Haiku 5.5) and a custom code check, run on single sessions, on batches, on a dataset of scenarios and on simulated conversations, and continuously with online evaluation. Results are in CloudWatch.

**Next:** [`2_agent_optimization.ipynb`](2_agent_optimization.ipynb). Use the traces to improve the orchestrator's prompt and tool descriptions.

## Step 13: Clean up
Delete the resources this notebook created. Batch and on-demand results stay in CloudWatch. The IAM roles stay in place because the online evaluation settings made in the console share them.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## \u26a0\ufe0f Uncomment and run this cell only after you are done with ALL notebooks.

# print("Deleting online eval config...")
# try:
#     agentcore_control.delete_online_evaluation_config(onlineEvaluationConfigId=ONLINE_CONFIG_ID)
#     print("  OK")
# except Exception as e:
#     print("  {}".format(e))
#
# # Wait for online config to release the custom evaluator locks
# time.sleep(15)
#
# for eid, name in [(LLM_EVALUATOR_ID, LLM_EVALUATOR_NAME),
#                   (CODE_EVALUATOR_ID, CODE_EVALUATOR_NAME)]:
#     print("Deleting evaluator {} ({})...".format(name, eid))
#     try:
#         agentcore_control.delete_evaluator(evaluatorId=eid)
#         print("  OK")
#     except Exception as e:
#         print("  {}".format(e))
#
# print("Deleting Lambda function {}...".format(LAMBDA_FUNCTION_NAME))
# try:
#     lambda_client.delete_function(FunctionName=LAMBDA_FUNCTION_NAME)
#     print("  OK")
# except Exception as e:
#     print("  {}".format(e))
#
# print("\nDone. IAM roles ({}, {}) left in place for future runs.".format(
#     EVAL_ROLE_NAME, LAMBDA_ROLE_NAME))